In [36]:
import requests
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
import os

In [32]:
pd.set_option("display.max_columns", None)   # muestra todas las columnas
pd.set_option("display.max_colwidth", None)  # no corta el contenido de texto largo (ej. tags, developers)
pd.set_option("display.width", None) 

In [18]:
load_dotenv()  # busca el archivo .env en la carpeta del proyecto
API_KEY = os.getenv("ITAD_API_KEY")
BASE = "https://api.isthereanydeal.com"

# FUNCION QUE OBTIENE A PARTIR DEL APPID DEL GAME STEAM  y de una fecha dada, el historico de precioos más su info

In [24]:
def get_game_data(appid, since="2015-01-01T00:00:00Z", country="us"):
    """Trae lookup + history + info para un appid de Steam."""

    r = requests.get(f"{BASE}/games/lookup/v1",
                      params={"key": API_KEY, "appid": appid})
    r.raise_for_status()
    lookup = r.json()
    if not lookup.get("found"):
        raise ValueError(f"Juego con appid {appid} no encontrado en ITAD")
    game_id = lookup["game"]["id"]
    title = lookup["game"]["title"]

    r = requests.get(f"{BASE}/games/history/v2",
                      params={"key": API_KEY, "id": game_id, "shops": 61,
                              "since": since, "country": country})
    r.raise_for_status()
    history = r.json()

    r = requests.get(f"{BASE}/games/info/v2",
                      params={"key": API_KEY, "id": game_id})
    r.raise_for_status()
    info = r.json()

    return {"appid": appid, "game_id": game_id, "title": title,
            "history": history, "info": info}

In [30]:
def info_to_dataframe(info):
    row = {
        "appid": info["appid"],
        "game_id": info["id"],
        "title": info["title"],
        "release_date": info.get("releaseDate"),
        "early_access": info.get("earlyAccess"),
        "achievements": info.get("achievements"),
        "trading_cards": info.get("tradingCards"),
        "tags": ", ".join(info.get("tags", [])),
        "developers": ", ".join(d["name"] for d in info.get("developers", [])),
        "publishers": ", ".join(p["name"] for p in info.get("publishers", [])),
        "players_recent": info.get("players", {}).get("recent"),
        "players_day": info.get("players", {}).get("day"),
        "players_week": info.get("players", {}).get("week"),
        "players_peak": info.get("players", {}).get("peak"),
        "itad_rank": info.get("stats", {}).get("rank"),
        "itad_waitlisted": info.get("stats", {}).get("waitlisted"),
        "itad_collected": info.get("stats", {}).get("collected"),
    }

    # Cada fuente de review (Steam, Metascore, etc.) se separa en sus propias columnas
    for review in info.get("reviews", []):
        source = review["source"].lower().replace(" ", "_")
        row[f"{source}_score"] = review["score"]
        row[f"{source}_count"] = review["count"]

    return pd.DataFrame([row])

In [25]:
result = get_game_data(3065800)
print(result["title"], "-", result["game_id"])
print(f"Registros de historial: {len(result['history'])}")

Marathon - 018d937f-221d-72ee-99a5-238681910c51
Registros de historial: 7


In [27]:
# Vista del historial como dataframe
df_hist = pd.DataFrame([{
    "timestamp": h["timestamp"],
    "price": h["deal"]["price"]["amount"],
    "regular": h["deal"]["regular"]["amount"],
    "discount_pct": h["deal"]["cut"]
} for h in result["history"]])
df_hist["timestamp"] = pd.to_datetime(df_hist["timestamp"], utc=True)
df_hist = df_hist.sort_values("timestamp").reset_index(drop=True)

In [28]:
df_hist

,timestamp,price,regular,discount_pct
0,2026-01-19 16:18:26+00:00,39.99,39.99,0
1,2026-04-21 17:18:58+00:00,31.99,39.99,20
2,2026-04-28 17:17:39+00:00,39.99,39.99,0
3,2026-06-02 17:17:02+00:00,27.99,39.99,30
4,2026-06-16 17:17:01+00:00,39.99,39.99,0
5,2026-06-25 18:56:53+00:00,27.99,39.99,30
6,2026-07-09 18:29:58+00:00,39.99,39.99,0


In [33]:
df_info = info_to_dataframe(result["info"])
df_info

,appid,game_id,title,release_date,early_access,achievements,trading_cards,tags,developers,publishers,players_recent,players_day,players_week,players_peak,itad_rank,itad_waitlisted,itad_collected,steam_score,steam_count,metascore_score,metascore_count,metacritic_user_score_score,metacritic_user_score_count,opencritic_score,opencritic_count
0,3065800,018d937f-221d-72ee-99a5-238681910c51,Marathon,2026-03-04,False,True,False,"Extraction Shooter, PvP, Multiplayer, Sci-fi, Shooter",Bungie,"Bungie, Inc, Bungie, Bungie Inc",4726,6435,6439,88337,3818,5285,1551,82,43398,81,47,57,1434,81,132


In [34]:
DATA_DIR = "data"
PRICE_HISTORY_PATH = os.path.join(DATA_DIR, "price_history.csv")
SNAPSHOTS_PATH = os.path.join(DATA_DIR, "snapshots.csv")

os.makedirs(DATA_DIR, exist_ok=True)


def save_price_history(result):
    """Guarda el histórico de precios sin duplicar filas ya existentes."""
    appid = result["appid"]
    game_id = result["game_id"]
    title = result["title"]

    df_new = pd.DataFrame([{
        "appid": appid,
        "game_id": game_id,
        "title": title,
        "timestamp": h["timestamp"],
        "shop": h["shop"]["name"],
        "price": h["deal"]["price"]["amount"],
        "regular": h["deal"]["regular"]["amount"],
        "discount_pct": h["deal"]["cut"]
    } for h in result["history"]])

    if df_new.empty:
        print(f"[{title}] Sin registros de historial nuevos para guardar.")
        return

    df_new["timestamp"] = pd.to_datetime(df_new["timestamp"], utc=True)

    if os.path.exists(PRICE_HISTORY_PATH):
        df_old = pd.read_csv(PRICE_HISTORY_PATH, parse_dates=["timestamp"])
        df_old["timestamp"] = pd.to_datetime(df_old["timestamp"], utc=True)
        df_all = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_all = df_new

    before = len(df_all)
    df_all = df_all.drop_duplicates(subset=["appid", "timestamp", "shop"])
    df_all = df_all.sort_values(["appid", "timestamp"]).reset_index(drop=True)
    after = len(df_all)

    df_all.to_csv(PRICE_HISTORY_PATH, index=False)
    print(f"[{title}] Historial guardado. Filas nuevas agregadas: {after - (before - len(df_new))}. Total en archivo: {after}")


def save_snapshot(result):
    """Guarda un snapshot diario de metadata (jugadores, reviews, rank, etc.)."""
    info = result["info"]
    today = datetime.now(timezone.utc).date().isoformat()

    row = {
        "appid": info["appid"],
        "game_id": info["id"],
        "title": info["title"],
        "date_checked": today,
        "release_date": info.get("releaseDate"),
        "tags": ", ".join(info.get("tags", [])),
        "developers": ", ".join(d["name"] for d in info.get("developers", [])),
        "players_recent": info.get("players", {}).get("recent"),
        "players_day": info.get("players", {}).get("day"),
        "players_week": info.get("players", {}).get("week"),
        "players_peak": info.get("players", {}).get("peak"),
        "itad_rank": info.get("stats", {}).get("rank"),
        "itad_waitlisted": info.get("stats", {}).get("waitlisted"),
        "itad_collected": info.get("stats", {}).get("collected"),
    }

    for review in info.get("reviews", []):
        source = review["source"].lower().replace(" ", "_")
        row[f"{source}_score"] = review["score"]
        row[f"{source}_count"] = review["count"]

    df_new = pd.DataFrame([row])

    if os.path.exists(SNAPSHOTS_PATH):
        df_old = pd.read_csv(SNAPSHOTS_PATH)
        df_all = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_all = df_new

    # Si ya corriste hoy para este juego, evita duplicar (se queda con la última)
    df_all = df_all.drop_duplicates(subset=["appid", "date_checked"], keep="last")
    df_all = df_all.sort_values(["appid", "date_checked"]).reset_index(drop=True)

    df_all.to_csv(SNAPSHOTS_PATH, index=False)
    print(f"[{info['title']}] Snapshot de {today} guardado. Total en archivo: {len(df_all)}")

In [37]:
appids = [3065800]  # agrega más appids aquí cuando quieras escalar

for appid in appids:
    result = get_game_data(appid)
    save_price_history(result)
    save_snapshot(result)

[Marathon] Historial guardado. Filas nuevas agregadas: 0. Total en archivo: 7
[Marathon] Snapshot de 2026-07-19 guardado. Total en archivo: 1


HTTPError: 404 Client Error: Not Found for url: https://api.steampowered.com/ISteamApps/GetAppList/v2/